In [7]:
import pandas as pd

# Load dataset
data = pd.read_csv("multiple_linear_regression_dataset.csv")

# Inspect data
print("First 5 rows:\n", data.head())
print("\nColumns:", data.columns)
print("\nShape:", data.shape)

# At this point, I think the inputs are "Years Experience" and "TestScore"
# because these are the independent variables we would know beforehand.
# The output is "Salary", which is what we want to predict.
# My assumption here is that my model will need 2 features (inputs) to work.



First 5 rows:
    age  experience  income
0   25           1   30450
1   30           3   35670
2   47           2   31580
3   32           5   40130
4   43          10   47830

Columns: Index(['age', 'experience', 'income'], dtype='object')

Shape: (20, 3)


In [9]:
# Inputs (features)
X = data[["age", "experience"]].values

# Output (target)
y = data["income"].values

# Thinking about shapes: X should be an N x 2 matrix (N samples, 2 features),
# and y should be an N x 1 vector. y only has one column because we are only
# trying to predict one specific continuous income value per person.
print("\nShape of X:", X.shape)
print("Shape of y:", y.shape)


Shape of X: (20, 2)
Shape of y: (20,)


In [10]:
import numpy as np

# Number of features
n_features = X.shape[1]

# Initialize weights and bias
w = np.zeros(n_features)
b = 0.0

# I am initializing w as an array of two zeros, and b as a single zero scalar.
# We need one weight per feature to learn the independent importance of
# Years Experience vs. TestScore.
# Bias is separate because it acts as the "base salary" if experience and score were both 0.
# If we initialized with huge random numbers, the initial error would be massive,
# which might make the gradients explode and the learning unstable.

In [11]:
def predict(X, w, b):
    # X: input features, w: weights, b: bias
    y_hat = X.dot(w) + b
    return y_hat

# Earlier I thought we always needed an activation function like Sigmoid (from Lab 3).
# But now I realize we drop it here because we want raw continuous numbers (like $65,000),
# not a probability squeezed between 0 and 1.

def mean_squared_error(y, y_hat):
    # y: actual values, y_hat: predicted values
    loss = ((y_hat - y) ** 2).mean()
    return loss

# We square the error to ensure negative and positive mistakes don't cancel each other out.
# Also, squaring heavily penalizes predictions that are VERY wrong.
# Taking the absolute error would treat all errors linearly, but MSE forces the model
# to care much more about fixing its biggest mistakes.

In [12]:
def compute_gradients(X, y, y_hat):
    N = len(y)

    # Gradient w.r.t weights: $$dw = \frac{2}{N} X^T (\hat{y} - y)$$
    dw = (2/N) * X.T.dot(y_hat - y)

    # Gradient w.r.t bias: $$db = \frac{2}{N} \sum (\hat{y} - y)$$
    db = (2/N) * (y_hat - y).sum()

    return dw, db

# X appears in dw because the size of the input feature dictates how much
# that specific weight contributed to the error. b is a constant addition, so X isn't in db.
# If the error (y_hat - y) is zero, dw and db become zero, meaning learning stops
# because the model is perfect!

def update_parameters(w, b, dw, db, lr):
    w = w - lr * dw
    b = b - lr * db
    return w, b

In [13]:
# Initial hyperparameter guesses
lr = 0.0001
epochs = 1000

print("Starting training...")
for epoch in range(epochs):
    # Predict
    y_hat = predict(X, w, b)

    # Calculate Loss
    loss = mean_squared_error(y, y_hat)

    # Get Gradients
    dw, db = compute_gradients(X, y, y_hat)

    # Update Weights
    w, b = update_parameters(w, b, dw, db, lr)

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.2f}")

# At first, I was worried if the loss would bounce around.
# But my assumption here is that with a small learning rate (0.0001),
# the loss should steadily decrease over time.
# If it increases, it means my learning rate is too high and the model is
# overshooting the minimum error. If epochs are too low, it stops learning too early.

Starting training...
Epoch 0, Loss: 1727049635.00
Epoch 100, Loss: 66491868.55
Epoch 200, Loss: 61752567.20
Epoch 300, Loss: 58616531.08
Epoch 400, Loss: 56528801.54
Epoch 500, Loss: 55126542.03
Epoch 600, Loss: 54172526.95
Epoch 700, Loss: 53511656.14
Epoch 800, Loss: 53042523.73
Epoch 900, Loss: 52698829.56


In [14]:
print("\n=== Final Model Parameters ===")
print(f"Weights (Age, Experience): {w}")
print(f"Bias: {b:.2f}")

# Let's test a new candidate not in the dataset
# Candidate is 28 years old with 5 years of experience
new_candidate = np.array([28, 5])
predicted_income = new_candidate.dot(w) + b

# Updated the text here to match the 28 years old and 5 years experience!
print(f"\nPredicted Income for candidate (28 yrs old, 5 yrs exp): ${predicted_income:.2f}")

# The prediction seems reasonable and interpolates smoothly between data points.
# Earlier I thought simple IF/THEN threshold rules were fine for basic logic.
# But now I realize this linear regression model is vastly superior for this task
# because it dynamically calculates an exact value based on continuous relationships,
# rather than clunky, hard-coded brackets.


=== Final Model Parameters ===
Weights (Age, Experience): [ 764.75405919 1371.03430441]
Bias: 321.74

Predicted Income for candidate (28 yrs old, 5 yrs exp): $28590.02
